## 1. Load Sample Sales Data

Import required libraries and load/generate sample sales data with date and sales columns.

In [ ]:
# Import required libraries
import sys
import os
sys.path.insert(0, '/home/petpooja/Enterprise Retail Intelligence System/backend')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Prophet and evaluation
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print('✅ All libraries imported successfully')

In [ ]:
# Load the sample data generator from the backend
from app.ml.data.generate_sales_data import SalesDataGenerator

# Generate sample sales data
generator = SalesDataGenerator(
    base_sales=150000,
    trend_rate=0.001,
    seasonality_strength=0.4,
    noise_level=0.15
)

# Generate 365 days of sales data
df = generator.generate(days=365)

# Display dataset info
print(f"Dataset shape: {df.shape}")
print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
print(f"\nBasic Statistics:")
print(df['sales'].describe())
print(f"\nFirst 5 rows:")
df.head()

## 2. Exploratory Data Analysis (EDA)

Create visualizations to explore the data, identify patterns, seasonality, and trends.

In [ ]:
# Time series plot
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Full time series
axes[0, 0].plot(df['date'], df['sales'], linewidth=1.5, color='steelblue')
axes[0, 0].set_title('Daily Sales Time Series (365 days)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Sales (₹)')
axes[0, 0].grid(True, alpha=0.3)

# 2. Sales distribution
axes[0, 1].hist(df['sales'], bins=40, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Sales Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Sales (₹)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(df['sales'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: ₹{df["sales"].mean():,.0f}')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Weekly seasonality (rolling mean)
df['rolling_7'] = df['sales'].rolling(window=7).mean()
df['rolling_30'] = df['sales'].rolling(window=30).mean()
axes[1, 0].plot(df['date'], df['sales'], alpha=0.3, label='Daily Sales', color='lightblue')
axes[1, 0].plot(df['date'], df['rolling_7'], linewidth=2, label='7-day Rolling Mean', color='steelblue')
axes[1, 0].plot(df['date'], df['rolling_30'], linewidth=2, label='30-day Rolling Mean', color='darkred')
axes[1, 0].set_title('Sales Trend with Rolling Averages', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Sales (₹)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Monthly aggregated sales
df['year_month'] = df['date'].dt.to_period('M')
monthly_sales = df.groupby('year_month')['sales'].sum()
axes[1, 1].bar(range(len(monthly_sales)), monthly_sales.values, color='teal', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Monthly Aggregated Sales', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Month')
axes[1, 1].set_ylabel('Total Sales (₹)')
axes[1, 1].set_xticklabels([str(m) for m in monthly_sales.index], rotation=45)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/home/petpooja/Enterprise Retail Intelligence System/research/notebooks/01_eda_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print('✅ EDA visualizations created')

In [ ]:
# Seasonality analysis - Day of week effect
df['day_of_week'] = df['date'].dt.day_name()
df['is_weekend'] = df['date'].dt.dayofweek >= 5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Day of week seasonality
dow_sales = df.groupby('day_of_week')['sales'].mean()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_sales = dow_sales.reindex(dow_order)

colors = ['steelblue' if day not in ['Saturday', 'Sunday'] else 'coral' for day in dow_order]
axes[0].bar(range(len(dow_sales)), dow_sales.values, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_title('Average Sales by Day of Week', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Day of Week')
axes[0].set_ylabel('Average Sales (₹)')
axes[0].set_xticklabels(dow_order, rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# Weekday vs Weekend
weekend_effect = df.groupby('is_weekend')['sales'].agg(['mean', 'std'])
weekend_effect.index = ['Weekday', 'Weekend']
weekend_effect['mean'].plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'], edgecolor='black', alpha=0.8)
axes[1].set_title('Weekday vs Weekend Sales', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Average Sales (₹)')
axes[1].set_xticklabels(['Weekday', 'Weekend'], rotation=0)
axes[1].grid(True, alpha=0.3, axis='y')

# Add error bars
for i, (idx, row) in enumerate(weekend_effect.iterrows()):
    axes[1].errorbar(i, row['mean'], yerr=row['std'], fmt='none', ecolor='black', capsize=5, capthick=2)

plt.tight_layout()
plt.savefig('/home/petpooja/Enterprise Retail Intelligence System/research/notebooks/02_seasonality_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

# Print seasonality statistics
print("\n📊 Seasonality Statistics:")
print(f"\nAverage sales by day of week:")
print(dow_sales)
print(f"\nWeekend effect: {((weekend_effect.loc['Weekend', 'mean'] / weekend_effect.loc['Weekday', 'mean']) - 1) * 100:.1f}% higher on weekends")

## 3. Train Prophet Model

Prepare data in Prophet format, instantiate and fit the Prophet model to training data.

In [ ]:
# Split data into training and testing sets (80/20 split)
train_size = int(len(df) * 0.8)
train_df = df[:train_size].copy()
test_df = df[train_size:].copy()

print(f"Training set: {len(train_df)} days ({train_df['date'].min().date()} to {train_df['date'].max().date()})")
print(f"Test set: {len(test_df)} days ({test_df['date'].min().date()} to {test_df['date'].max().date()})")

# Prepare data for Prophet (requires 'ds' and 'y' columns)
prophet_train = train_df[['date', 'sales']].copy()
prophet_train.columns = ['ds', 'y']

# Add regressors (features)
prophet_train['is_weekend'] = train_df['is_weekend'].astype(int)

# Add month for seasonality
prophet_train['month'] = prophet_train['ds'].dt.month

print(f"\n✅ Data prepared for Prophet")
print(f"\nTraining data shape: {prophet_train.shape}")
print(prophet_train.head())

In [ ]:
# Initialize and train Prophet model
print("🔄 Training Prophet model...")

# Create model with custom settings
model = Prophet(
    interval_width=0.95,  # 95% confidence intervals
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='multiplicative',  # Seasonality scales with trend
    changepoint_prior_scale=0.05
)

# Add regressors
model.add_regressor('is_weekend')
model.add_regressor('month')

# Add Indian holidays
holidays = pd.DataFrame({
    'holiday': ['Republic Day', 'Holi', 'Good Friday', 'Diwali', 'New Year', 
                'Independence Day', 'Gandhi Jayanti', 'Christmas'],
    'ds': pd.to_datetime(['2023-01-26', '2023-03-07', '2023-04-07', '2023-11-12',
                          '2023-01-01', '2023-08-15', '2023-10-02', '2023-12-25']),
    'lower_window': [-1],
    'upper_window': [1]
})

model.add_country_holidays(country_name='IN')

# Fit the model
import time
start_time = time.time()
model.fit(prophet_train)
training_time = time.time() - start_time

print(f"✅ Model trained successfully in {training_time:.2f} seconds")

## 4. Evaluate on Test Set

Make predictions on the test set and compare forecasted values against actual observed values.

In [ ]:
# Prepare test data with regressors
prophet_test = test_df[['date', 'sales']].copy()
prophet_test.columns = ['ds', 'y']
prophet_test['is_weekend'] = test_df['is_weekend'].astype(int)
prophet_test['month'] = prophet_test['ds'].dt.month

# Make predictions on test set
print("🔄 Making predictions on test set...")
forecast_test = model.predict(prophet_test[['ds', 'is_weekend', 'month']])

# Merge actual and predicted values
results = prophet_test[['ds', 'y']].copy()
results['yhat'] = forecast_test['yhat'].values
results['yhat_lower'] = forecast_test['yhat_lower'].values
results['yhat_upper'] = forecast_test['yhat_upper'].values

# Calculate residuals
results['residual'] = results['y'] - results['yhat']
results['absolute_error'] = np.abs(results['residual'])
results['percentage_error'] = np.abs(results['residual'] / results['y'] * 100)

print(f"✅ Predictions generated for {len(results)} test samples")
print(f"\nFirst 5 predictions:")
print(results[['ds', 'y', 'yhat', 'yhat_lower', 'yhat_upper']].head())
print(f"\nLast 5 predictions:")
print(results[['ds', 'y', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

## 5. Plot Predictions

Visualize Prophet forecasts including trend, seasonality components, and confidence intervals.

In [ ]:
# Plot 1: Test set predictions vs actual
fig, ax = plt.subplots(figsize=(14, 6))

# Plot actual values
ax.plot(results['ds'], results['y'], 'o-', linewidth=2, label='Actual Sales', color='darkblue', markersize=3)

# Plot predictions
ax.plot(results['ds'], results['yhat'], 's--', linewidth=2, label='Prophet Forecast', color='coral', markersize=3)

# Plot confidence intervals
ax.fill_between(results['ds'], results['yhat_lower'], results['yhat_upper'], 
                  alpha=0.2, color='coral', label='95% Confidence Interval')

ax.set_title('Prophet Forecast vs Actual Sales (Test Set)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Sales (₹)')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/petpooja/Enterprise Retail Intelligence System/research/notebooks/03_test_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

print('✅ Test predictions plot created')

In [ ]:
# Plot 2: Full forecast (future dates)
future = model.make_future_dataframe(periods=30)  # 30-day forecast
future['is_weekend'] = future['ds'].dt.dayofweek >= 5
future['month'] = future['ds'].dt.month
future['is_weekend'] = future['is_weekend'].astype(int)

forecast = model.predict(future)

fig, ax = plt.subplots(figsize=(14, 6))

# Plot historical data
ax.plot(df['date'], df['sales'], 'o-', linewidth=1.5, label='Historical Sales', color='steelblue', markersize=2)

# Plot forecast
forecast_future = forecast[forecast['ds'] > df['date'].max()]
ax.plot(forecast_future['ds'], forecast_future['yhat'], 's-', linewidth=2, label='30-Day Forecast', color='coral')

# Plot confidence intervals for future
ax.fill_between(forecast_future['ds'], forecast_future['yhat_lower'], forecast_future['yhat_upper'], 
                  alpha=0.3, color='coral', label='95% Confidence Interval')

ax.set_title('Historical Sales & 30-Day Future Forecast', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Sales (₹)')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/petpooja/Enterprise Retail Intelligence System/research/notebooks/04_future_forecast.png', dpi=100, bbox_inches='tight')
plt.show()

print('✅ Future forecast plot created')

In [ ]:
# Plot 3: Prophet components (trend and seasonality)
fig = model.plot_components(forecast, figsize=(14, 10))
plt.tight_layout()
plt.savefig('/home/petpooja/Enterprise Retail Intelligence System/research/notebooks/05_prophet_components.png', dpi=100, bbox_inches='tight')
plt.show()

print('✅ Prophet components plot created')

## 6. Analyze Residuals

Calculate and analyze residuals to check for patterns, autocorrelation, and model assumptions.

In [ ]:
# Residual analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals over time
axes[0, 0].plot(results['ds'], results['residual'], 'o-', color='steelblue', markersize=3, linewidth=1)
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_title('Residuals Over Time', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Residual (₹)')
axes[0, 0].grid(True, alpha=0.3)

# 2. Residuals distribution
axes[0, 1].hist(results['residual'], bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(results['residual'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: ₹{results["residual"].mean():,.0f}')
axes[0, 1].axvline(results['residual'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: ₹{results["residual"].median():,.0f}')
axes[0, 1].set_title('Residuals Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Residual (₹)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Q-Q Plot for normality
from scipy import stats
stats.probplot(results['residual'], dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Normality Check)', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# 4. ACF plot for autocorrelation
plot_acf(results['residual'], lags=20, ax=axes[1, 1])
axes[1, 1].set_title('ACF Plot (Autocorrelation)', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/petpooja/Enterprise Retail Intelligence System/research/notebooks/06_residuals_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print('✅ Residuals analysis plots created')

In [ ]:
# Residuals statistics
print("\n📊 Residual Statistics:")
print(f"Mean: ₹{results['residual'].mean():,.2f}")
print(f"Std Dev: ₹{results['residual'].std():,.2f}")
print(f"Min: ₹{results['residual'].min():,.2f}")
print(f"Max: ₹{results['residual'].max():,.2f}")
print(f"25th percentile: ₹{results['residual'].quantile(0.25):,.2f}")
print(f"Median: ₹{results['residual'].median():,.2f}")
print(f"75th percentile: ₹{results['residual'].quantile(0.75):,.2f}")

# Normality test
from scipy.stats import shapiro, normaltest
stat, p_value = normaltest(results['residual'])
print(f"\nNormality Test (D'Agostino-Pearson):")
print(f"Test Statistic: {stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Result: {'Residuals are normally distributed' if p_value > 0.05 else 'Residuals may not be normally distributed'} (α=0.05)")

## 7. Calculate Performance Metrics

Compute evaluation metrics including RMSE, MAE, MAPE, and direction accuracy.

In [ ]:
# Calculate performance metrics
print("\n" + "="*60)
print("📊 PROPHET MODEL PERFORMANCE METRICS")
print("="*60)

# 1. RMSE (Root Mean Squared Error)
rmse = np.sqrt(mean_squared_error(results['y'], results['yhat']))
print(f"\n1. RMSE (Root Mean Squared Error):")
print(f"   Value: ₹{rmse:,.2f}")
print(f"   Interpretation: Average prediction error in absolute terms")

# 2. MAE (Mean Absolute Error)
mae = mean_absolute_error(results['y'], results['yhat'])
print(f"\n2. MAE (Mean Absolute Error):")
print(f"   Value: ₹{mae:,.2f}")
print(f"   Interpretation: Average absolute deviation from actual values")

# 3. MAPE (Mean Absolute Percentage Error)
mape = results['percentage_error'].mean()
print(f"\n3. MAPE (Mean Absolute Percentage Error):")
print(f"   Value: {mape:.2f}%")
print(f"   Interpretation: Average percentage error across all predictions")

# 4. Direction Accuracy
actual_direction = np.diff(results['y'].values) > 0
predicted_direction = np.diff(results['yhat'].values) > 0
direction_accuracy = np.mean(actual_direction == predicted_direction) * 100
print(f"\n4. Direction Accuracy:")
print(f"   Value: {direction_accuracy:.2f}%")
print(f"   Interpretation: Percentage of correctly predicted trend direction")

# 5. R-squared (Coefficient of Determination)
ss_res = np.sum((results['y'] - results['yhat']) ** 2)
ss_tot = np.sum((results['y'] - results['y'].mean()) ** 2)
r_squared = 1 - (ss_res / ss_tot)
print(f"\n5. R-squared (Coefficient of Determination):")
print(f"   Value: {r_squared:.4f}")
print(f"   Interpretation: Proportion of variance explained by the model")

# 6. MAPE by percentiles
print(f"\n6. MAPE Distribution:")
print(f"   Mean: {mape:.2f}%")
print(f"   Median: {results['percentage_error'].median():.2f}%")
print(f"   Std Dev: {results['percentage_error'].std():.2f}%")
print(f"   Min: {results['percentage_error'].min():.2f}%")
print(f"   Max: {results['percentage_error'].max():.2f}%")
print(f"   25th percentile: {results['percentage_error'].quantile(0.25):.2f}%")
print(f"   75th percentile: {results['percentage_error'].quantile(0.75):.2f}%")

print("\n" + "="*60)

In [ ]:
# Create metrics visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Key metrics comparison
metrics_data = {
    'RMSE (₹K)': rmse / 1000,
    'MAE (₹K)': mae / 1000,
    'MAPE (%)': mape,
    'R² (×100)': r_squared * 100,
    'Dir Acc (%)': direction_accuracy
}

colors_metrics = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']
axes[0, 0].bar(metrics_data.keys(), metrics_data.values(), color=colors_metrics, edgecolor='black', alpha=0.8)
axes[0, 0].set_title('Key Performance Metrics', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Value')
axes[0, 0].grid(True, alpha=0.3, axis='y')
axes[0, 0].tick_params(axis='x', rotation=45)

# Add value labels on bars
for i, (k, v) in enumerate(metrics_data.items()):
    axes[0, 0].text(i, v + max(metrics_data.values())*0.02, f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

# 2. Actual vs Predicted scatter
axes[0, 1].scatter(results['y'], results['yhat'], alpha=0.6, s=30, color='steelblue', edgecolor='black')
min_val = min(results['y'].min(), results['yhat'].min())
max_val = max(results['y'].max(), results['yhat'].max())
axes[0, 1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Fit')
axes[0, 1].set_xlabel('Actual Sales (₹)')
axes[0, 1].set_ylabel('Predicted Sales (₹)')
axes[0, 1].set_title('Actual vs Predicted Sales', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Absolute error distribution
axes[1, 0].hist(results['absolute_error'], bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[1, 0].axvline(results['absolute_error'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: ₹{results["absolute_error"].mean():,.0f}')
axes[1, 0].set_title('Absolute Error Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Absolute Error (₹)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. MAPE over time
axes[1, 1].plot(results['ds'], results['percentage_error'], 'o-', color='steelblue', markersize=3, linewidth=1)
axes[1, 1].axhline(y=mape, color='red', linestyle='--', linewidth=2, label=f'Mean MAPE: {mape:.2f}%')
axes[1, 1].fill_between(results['ds'], 0, results['percentage_error'], alpha=0.2, color='steelblue')
axes[1, 1].set_title('MAPE Over Time', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('MAPE (%)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/petpooja/Enterprise Retail Intelligence System/research/notebooks/07_metrics_summary.png', dpi=100, bbox_inches='tight')
plt.show()

print('✅ Metrics summary plots created')

In [ ]:
# Summary statistics table
summary_df = pd.DataFrame({
    'Metric': ['Training Samples', 'Test Samples', 'Training Time (sec)', 'RMSE (₹)', 'MAE (₹)', 
               'MAPE (%)', 'Direction Accuracy (%)', 'R-squared'],
    'Value': [
        f"{len(train_df)}",
        f"{len(test_df)}",
        f"{training_time:.2f}",
        f"₹{rmse:,.2f}",
        f"₹{mae:,.2f}",
        f"{mape:.2f}%",
        f"{direction_accuracy:.2f}%",
        f"{r_squared:.4f}"
    ]
})

print("\n📋 Summary Table:")
print(summary_df.to_string(index=False))

# Save results to CSV
results.to_csv('/home/petpooja/Enterprise Retail Intelligence System/research/notebooks/test_predictions.csv', index=False)
summary_df.to_csv('/home/petpooja/Enterprise Retail Intelligence System/research/notebooks/metrics_summary.csv', index=False)
print("\n✅ Results saved to CSV files")

## 8. Conclusions and Insights

Summary of findings from the Prophet forecasting analysis.

In [ ]:
print("\n" + "="*70)
print("📈 FORECASTING ANALYSIS - KEY INSIGHTS")
print("="*70)

print(f"\n✅ MODEL PERFORMANCE:")
print(f"   - MAPE of {mape:.2f}% indicates {'excellent' if mape < 10 else 'good' if mape < 20 else 'acceptable' if mape < 30 else 'moderate'} forecast accuracy")
print(f"   - Direction Accuracy of {direction_accuracy:.1f}% shows {'strong' if direction_accuracy > 70 else 'moderate'} trend prediction capability")
print(f"   - R² of {r_squared:.4f} explains {r_squared*100:.2f}% of sales variance")

print(f"\n📊 DATA PATTERNS IDENTIFIED:")
print(f"   - Weekend Effect: {((weekend_effect.loc['Weekend', 'mean'] / weekend_effect.loc['Weekday', 'mean']) - 1) * 100:.1f}% higher sales")
print(f"   - Trend: {'Upward' if train_df['sales'].iloc[-1] > train_df['sales'].iloc[0] else 'Downward'} trend over {len(train_df)} days")
print(f"   - Volatility: ₹{train_df['sales'].std():,.2f} standard deviation")

print(f"\n🎯 MODEL STRENGTHS:")
print(f"   - Captures seasonality (weekly and yearly patterns)")
print(f"   - Incorporates external regressors (weekends)")
print(f"   - Provides confidence intervals for risk assessment")
print(f"   - Handles holidays and special events")

print(f"\n⚠️ AREAS FOR IMPROVEMENT:")
if mape > 25:
    print(f"   - MAPE of {mape:.2f}% could be improved by adding more regressors")
if direction_accuracy < 70:
    print(f"   - Direction accuracy of {direction_accuracy:.1f}% suggests exploring ensemble methods")
if results['residual'].std() > results['y'].std() * 0.3:
    print(f"   - High residual variance may benefit from alternative models")

print(f"\n💡 RECOMMENDATIONS:")
print(f"   1. Monitor forecast accuracy on fresh data regularly")
print(f"   2. Consider adding external regressors (promotions, weather, etc.)")
print(f"   3. Retrain model monthly with latest sales data")
print(f"   4. Compare with ARIMA/SARIMA for ensemble forecasting")
print(f"   5. Investigate anomalies and their business context")

print("\n" + "="*70)